<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/fluidflow/neqsim_fenicsx_fem_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeqSim + FEniCSx: local FEM from process-model boundary conditions

This advanced example connects **NeqSim Java master → fluid properties / hydrate equilibrium → 1D pipeline screen → FEniCSx axisymmetric wall/insulation FEM → cooldown / hydrate margin → thermo-elastic stress**. The case is a wet-gas subsea line with a local degraded-insulation patch. It is a teaching and screening model, not a piping-code assessment.

A deliberate process boundary is used between the two numerical engines: NeqSim runs in a short isolated Python/JVM subprocess and writes a JSON handoff. FEniCSx then runs in the notebook process without a JVM loaded. This mirrors a robust engineering workflow in which process-model results become explicit boundary data for a local FEM model.

In [ ]:
import hashlib, importlib.util, os, shutil, subprocess, sys
from pathlib import Path

NEQSIM_SOURCE_REF = 'master'
os.environ['NEQSIM_JVM_AUTOSTART'] = '0'

def rq(cmd, cwd=None):
    return subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=True).stdout

rq([sys.executable, '-m', 'pip', 'install', '-q', 'neqsim', 'scipy'])
if importlib.util.find_spec('dolfinx') is None:
    installer = Path('/tmp/fenicsx.sh')
    rq(['wget', '-q', 'https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh', '-O', str(installer)])
    rq(['bash', str(installer)])

src = Path('/content/neqsim-java')
if src.exists():
    shutil.rmtree(src)
rq(['git', 'clone', '--depth', '1', '--branch', NEQSIM_SOURCE_REF, 'https://github.com/equinor/neqsim.git', str(src)])
neqsim_commit = rq(['git', '-C', str(src), 'rev-parse', 'HEAD']).strip()
rq(['./mvnw', '-q', '-DskipTests', '-P', 'shade', 'package'], cwd=src)
candidates = [p for p in (src / 'target').glob('neqsim-*.jar') if '-sources' not in p.name and '-javadoc' not in p.name and not p.name.startswith('original-')]
if not candidates:
    raise FileNotFoundError('No NeqSim runtime JAR found')
neqsim_jar = max(candidates, key=lambda p: p.stat().st_size)
assert neqsim_jar.stat().st_size > 5_000_000
neqsim_jar_sha256 = hashlib.sha256(neqsim_jar.read_bytes()).hexdigest()

print('NeqSim master commit:', neqsim_commit)
print('NeqSim runtime JAR:', neqsim_jar)
print('JAR SHA-256:', neqsim_jar_sha256)
print('FEniCSx installed:', importlib.util.find_spec('dolfinx') is not None)

## 1. NeqSim fluid → explicit FEM handoff

NeqSim supplies phase equilibrium, density, viscosity, thermal conductivity and mass-specific heat capacity. A Gnielinski correlation converts these into an internal convection coefficient, and a state-updated 1D energy balance establishes the bulk-gas temperature at the local FEM window.

The same isolated NeqSim process calculates hydrate equilibrium and a 48-hour bulk cooldown with `SurfCooldownAnalyzer`. The complete time history is serialized before the Java process exits.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Tin, Pin, Pout, Tsea = 60.0, 80.0, 70.0, 4.0
Di, ts, ti = 0.254, 0.0127, 0.050
ri, rs, ro = Di / 2, Di / 2 + ts, Di / 2 + ts + ti
Lpipe, velocity = 20_000.0, 5.0
ks, kins, kbad = 50.0, 0.17, 0.70
ho = 300.0
localL, z0, z1 = 4.0, 1.5, 2.5
rhos, cps, rhoi, cpi = 7850.0, 500.0, 600.0, 1700.0
depth, rhosea, g = 300.0, 1025.0, 9.80665

neqsim_worker = r'''
import json, math, os
from pathlib import Path
import jpype

jar = os.environ['NEQSIM_JAR']
out = Path(os.environ['NEQSIM_RESULTS'])
jpype.startJVM('-Xrs', classpath=[jar], convertStrings=False, interrupt=False)
SystemSrkCPAstatoil = jpype.JClass('neqsim.thermo.system.SystemSrkCPAstatoil')
ThermodynamicOperations = jpype.JClass('neqsim.thermodynamicoperations.ThermodynamicOperations')
SurfCooldownAnalyzer = jpype.JClass('neqsim.pvtsimulation.flowassurance.SurfCooldownAnalyzer')
source_location = str(SurfCooldownAnalyzer.class_.getProtectionDomain().getCodeSource().getLocation())
assert Path(jar).name in source_location

Tin, Pin, Pout, Tsea = 60.0, 80.0, 70.0, 4.0
Di, ts, ti = 0.254, 0.0127, 0.050
ri, rs, ro = Di/2, Di/2+ts, Di/2+ts+ti
Lpipe, velocity, ks, kins, ho = 20000.0, 5.0, 50.0, 0.17, 300.0
dry = {'nitrogen':0.01, 'CO2':0.02, 'methane':0.85, 'ethane':0.07, 'propane':0.03, 'i-butane':0.006, 'n-butane':0.008, 'i-pentane':0.002, 'n-pentane':0.002, 'n-hexane':0.002}
water = 2e-4
composition = {name: value*(1.0-water) for name, value in dry.items()}
composition['water'] = water

def build_fluid(Tc, Pbara, hydrate=False):
    fluid = SystemSrkCPAstatoil(Tc + 273.15, Pbara)
    for name, value in composition.items():
        fluid.addComponent(name, float(value))
    fluid.setMixingRule(10)
    fluid.setMultiPhaseCheck(True)
    fluid.setHydrateCheck(bool(hydrate))
    ThermodynamicOperations(fluid).TPflash()
    fluid.initPhysicalProperties()
    return fluid

def props(Tc, Pbara):
    phase = build_fluid(Tc, Pbara).getPhase('gas')
    return {'rho':float(phase.getDensity('kg/m3')), 'mu':float(phase.getViscosity('kg/msec')), 'k':float(phase.getThermalConductivity('W/mK')), 'cp':float(phase.getCp('J/kgK'))}

p0 = props(Tin, Pin)
Re = p0['rho']*velocity*Di/p0['mu']
Pr = p0['cp']*p0['mu']/p0['k']
friction = (0.79*math.log(Re)-1.64)**-2
Nu = (friction/8.0)*(Re-1000.0)*Pr/(1.0+12.7*math.sqrt(friction/8.0)*(Pr**(2.0/3.0)-1.0))
hi = Nu*p0['k']/Di
def resistance():
    return 1.0/(hi*2.0*math.pi*ri) + math.log(rs/ri)/(2.0*math.pi*ks) + math.log(ro/rs)/(2.0*math.pi*kins) + 1.0/(ho*2.0*math.pi*ro)

n = 121
x = [Lpipe*i/(n-1) for i in range(n)]
P = [Pin+(Pout-Pin)*i/(n-1) for i in range(n)]
T = [Tin]
area = math.pi*ri*ri
for j in range(n-1):
    state = props(T[-1], P[j])
    mdot = state['rho']*velocity*area
    step = x[j+1]-x[j]
    T.append(T[-1]-(1.0/resistance())*(T[-1]-Tsea)/(mdot*state['cp'])*step)
mid = (n-1)//2
Tb, Pb = T[mid], P[mid]
gp = props(Tb, Pb)

hyd = build_fluid(Tb, Pb, True)
ThermodynamicOperations(hyd).hydrateFormationTemperature()
hydrate_temperature_c = float(hyd.getTemperature('C'))

screen = SurfCooldownAnalyzer(build_fluid(Tb, Pb))
screen.setInternalDiameter(Di); screen.setWallThickness(ts); screen.setInsulationThickness(ti)
screen.setInsulationConductivity(kins); screen.setExternalHTC(ho); screen.setSeabedTemperature(Tsea)
screen.setOperatingTemperature(Tb); screen.setHydrateMargin(3.0); screen.setTotalTimeHours(48.0); screen.setTimeStepMinutes(5.0)
screen.calculate()
cool = screen.getCooldownCalculator()
screen_time_h = [0.5*i for i in range(97)]
screen_temp_c = [float(cool.getTemperatureAtTime(t)-273.15) for t in screen_time_h]

result = {'source_location':source_location, 'p0':p0, 'gp':gp, 'Re':Re, 'Pr':Pr, 'Nu':Nu, 'hi':hi, 'x':x, 'P':P, 'T':T, 'Tb':Tb, 'Pb':Pb, 'hydrate_temperature_c':hydrate_temperature_c, 'screen_no_touch_h':float(screen.getNoTouchTimeHours()), 'screen_time_constant_h':float(screen.getTimeConstantHours()), 'screen_time_h':screen_time_h, 'screen_temp_c':screen_temp_c}
out.write_text(json.dumps(result, indent=2))
'''

handoff_path = Path('/tmp/neqsim_fenicsx_handoff.json')
env = os.environ.copy(); env['NEQSIM_JAR'] = str(neqsim_jar); env['NEQSIM_RESULTS'] = str(handoff_path)
subprocess.run([sys.executable, '-c', neqsim_worker], env=env, check=True)
handoff = json.loads(handoff_path.read_text())
p0, gp = handoff['p0'], handoff['gp']
Re, Pr, Nu, hi = handoff['Re'], handoff['Pr'], handoff['Nu'], handoff['hi']
x, P, T = np.array(handoff['x']), np.array(handoff['P']), np.array(handoff['T'])
Tb, Pb = handoff['Tb'], handoff['Pb']
Thyd, screen_no_touch = handoff['hydrate_temperature_c'], handoff['screen_no_touch_h']
screen_time_h, screen_temp_c = np.array(handoff['screen_time_h']), np.array(handoff['screen_temp_c'])
def Rprime(k_ins=kins): return 1/(hi*2*np.pi*ri)+np.log(rs/ri)/(2*np.pi*ks)+np.log(ro/rs)/(2*np.pi*k_ins)+1/(ho*2*np.pi*ro)
assert Path(neqsim_jar).name in handoff['source_location']
assert np.all(np.diff(screen_temp_c) <= 1e-10)
summary = pd.DataFrame({'rho [kg/m3]':[p0['rho']], 'mu [Pa s]':[p0['mu']], 'k [W/mK]':[p0['k']], 'Cp [J/kgK]':[p0['cp']], 'Re':[Re], 'Pr':[Pr], 'hi [W/m2K]':[hi]})
display(summary)
print('Main-only NeqSim source:', handoff['source_location'])
print(f'10 km FEM boundary: {Tb:.3f} degC, {Pb:.2f} bara')
print(f'Hydrate equilibrium: {Thyd:.3f} degC; NeqSim no-touch: {screen_no_touch:.2f} h')
plt.plot(x/1000, T); plt.xlabel('Distance [km]'); plt.ylabel('Bulk-gas temperature [degC]'); plt.grid(); plt.show()
plt.plot(screen_time_h, screen_temp_c); plt.axhline(Thyd+3, ls='--'); plt.xlabel('Shutdown time [h]'); plt.ylabel('NeqSim bulk temperature [degC]'); plt.grid(); plt.show()

## 2. FEniCSx local thermal field

The JVM handoff is complete before this point. The local $z-r$ model is axisymmetric, so each weak-form integral is weighted by $2\pi r$. The inner steel wall convects to the NeqSim bulk gas and the outer insulation convects to seawater. A one-metre section uses increased effective insulation conductivity to represent a local degradation/water-ingress sensitivity.

The far field is checked against the independent analytical cylindrical-resistance solution.

In [ ]:
from mpi4py import MPI
from petsc4py import PETSc
import dolfinx
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem
print('DOLFINx:', dolfinx.__version__)
assert MPI.COMM_WORLD.size == 1

def tags_rect(domain, rmin, rmax):
    fdim = domain.topology.dim-1
    inner_facets = mesh.locate_entities_boundary(domain, fdim, lambda xx: np.isclose(xx[1],rmin))
    outer_facets = mesh.locate_entities_boundary(domain, fdim, lambda xx: np.isclose(xx[1],rmax))
    entities = np.hstack([inner_facets,outer_facets]).astype(np.int32)
    values = np.hstack([np.ones(len(inner_facets),np.int32),2*np.ones(len(outer_facets),np.int32)])
    order = np.argsort(entities)
    return mesh.meshtags(domain, fdim, entities[order], values[order])

domain_t = mesh.create_rectangle(MPI.COMM_WORLD, np.array([[0.0,ri],[localL,ro]]), [100,24], cell_type=mesh.CellType.triangle)
thermal_tags = tags_rect(domain_t,ri,ro)
ds_t = ufl.Measure('ds',domain=domain_t,subdomain_data=thermal_tags); dx_t = ufl.Measure('dx',domain=domain_t)
Vt = fem.functionspace(domain_t,('Lagrange',1)); u,w = ufl.TrialFunction(Vt),ufl.TestFunction(Vt)
X = ufl.SpatialCoordinate(domain_t); r=X[1]
base_k = ufl.conditional(ufl.le(r,rs),ks,kins)
degraded = ufl.And(ufl.And(ufl.ge(X[0],z0),ufl.le(X[0],z1)),ufl.gt(r,rs))
kval = ufl.conditional(degraded,kbad,base_k)
a = kval*ufl.inner(ufl.grad(u),ufl.grad(w))*2*np.pi*r*dx_t + hi*u*w*2*np.pi*r*ds_t(1)+ho*u*w*2*np.pi*r*ds_t(2)
Lform = hi*(Tb+273.15)*w*2*np.pi*r*ds_t(1)+ho*(Tsea+273.15)*w*2*np.pi*r*ds_t(2)
thermal_problem = LinearProblem(a,Lform,petsc_options_prefix='heat_main_',petsc_options={'ksp_type':'cg','pc_type':'jacobi','ksp_rtol':1e-10})
Th = thermal_problem.solve(); assert thermal_problem.solver.getConvergedReason()>0
coords = Vt.tabulate_dof_coordinates(); Tc=Th.x.array-273.15
inner=np.isclose(coords[:,1],ri); z_inner,T_inner=coords[inner,0],Tc[inner]; order=np.argsort(z_inner)
q_uniform=(Tb-Tsea)/Rprime(); T_analytic=Tb-q_uniform/(hi*2*np.pi*ri)
far=inner & ((coords[:,0]<0.55)|(coords[:,0]>3.45)); T_far=float(Tc[far].mean())
assert Tsea-1e-6 <= float(Tc.min()) <= float(Tc.max()) <= Tb+1e-6
assert abs(T_far-T_analytic)<1.5
assert float(T_inner.min())<T_far
print(f'Analytical uniform inner-wall T: {T_analytic:.3f} degC')
print(f'FEM far-field inner-wall T: {T_far:.3f} degC')
print(f'FEM degraded-section minimum: {float(T_inner.min()):.3f} degC')
plt.plot(z_inner[order],T_inner[order],label='FEniCSx inner wall'); plt.axvspan(z0,z1,alpha=.2,label='degraded insulation'); plt.axhline(T_analytic,ls='--',label='uniform analytical'); plt.xlabel('Local z [m]'); plt.ylabel('Inner-wall T [degC]'); plt.legend(); plt.grid(); plt.show()
plt.figure(figsize=(10,3)); sc=plt.scatter(coords[:,0],coords[:,1],c=Tc,s=10); plt.colorbar(sc,label='Temperature [degC]'); plt.xlabel('Local z [m]'); plt.ylabel('Radius [m]'); plt.title('FEniCSx local temperature field'); plt.show()

## 3. Local transient refinement of NeqSim cooldown

The NeqSim `SurfCooldownAnalyzer` time history is the **time-dependent inner thermal boundary**. FEniCSx does not recalculate the whole pipeline cooldown; instead it resolves how the local steel/insulation field responds, including the degraded-insulation cold spot. This preserves a clear process-model → local-FEM hierarchy.

In [ ]:
target = Thyd+3.0

def local_cooldown(dt=1800.0,hours=48.0):
    domain=mesh.create_rectangle(MPI.COMM_WORLD,np.array([[0.0,ri],[localL,ro]]),[50,14],cell_type=mesh.CellType.triangle)
    tags=tags_rect(domain,ri,ro); ds=ufl.Measure('ds',domain=domain,subdomain_data=tags); dx=ufl.Measure('dx',domain=domain)
    V=fem.functionspace(domain,('Lagrange',1)); X=ufl.SpatialCoordinate(domain); r=X[1]
    base_k=ufl.conditional(ufl.le(r,rs),ks,kins); degraded=ufl.And(ufl.And(ufl.ge(X[0],z0),ufl.le(X[0],z1)),ufl.gt(r,rs)); kval=ufl.conditional(degraded,kbad,base_k)
    capacity=ufl.conditional(ufl.le(r,rs),rhos*cps,rhoi*cpi)
    previous=fem.Function(V); previous.x.array[:]=Tb+273.15
    u,w=ufl.TrialFunction(V),ufl.TestFunction(V); bulkK=fem.Constant(domain,PETSc.ScalarType(Tb+273.15)); seaK=fem.Constant(domain,PETSc.ScalarType(Tsea+273.15))
    a=capacity*u*w*2*np.pi*r*dx + dt*kval*ufl.inner(ufl.grad(u),ufl.grad(w))*2*np.pi*r*dx + dt*hi*u*w*2*np.pi*r*ds(1)+dt*ho*u*w*2*np.pi*r*ds(2)
    Lform=capacity*previous*w*2*np.pi*r*dx + dt*hi*bulkK*w*2*np.pi*r*ds(1)+dt*ho*seaK*w*2*np.pi*r*ds(2)
    problem=LinearProblem(a,Lform,petsc_options_prefix='cooldown_local_',petsc_options={'ksp_type':'cg','pc_type':'jacobi','ksp_rtol':1e-9})
    times=[0.0]; bulk=[Tb]; wall=[Tb]; coords_local=V.tabulate_dof_coordinates(); inner_local=np.isclose(coords_local[:,1],ri)
    for n in range(int(hours*3600/dt)):
        tnext=(n+1)*dt/3600.0
        bnext=float(np.interp(tnext,screen_time_h,screen_temp_c))
        bulkK.value=PETSc.ScalarType(bnext+273.15)
        current=problem.solve(); assert problem.solver.getConvergedReason()>0
        times.append(tnext); bulk.append(bnext); wall.append(float((current.x.array-273.15)[inner_local].min()))
        previous.x.array[:]=current.x.array
    return np.array(times),np.array(bulk),np.array(wall),problem

tc,bc,wc,cooldown_problem=local_cooldown()
def crossing(series):
    idx=np.where(series<=target)[0]
    return tc[idx[0]] if len(idx) else np.inf
assert np.all(np.diff(bc)<=1e-10)
print(f'NeqSim uniform-pipe no-touch: {screen_no_touch:.2f} h')
print(f'Interpolated NeqSim bulk crossing: {crossing(bc):.2f} h')
print(f'Local FEM wall crossing: {crossing(wc):.2f} h')
plt.plot(tc,bc,label='NeqSim bulk boundary'); plt.plot(tc,wc,label='FEniCSx minimum inner wall'); plt.axhline(target,ls='--',label='hydrate + 3 K'); plt.xlabel('Time [h]'); plt.ylabel('Temperature [degC]'); plt.legend(); plt.grid(); plt.show()

## 4. Thermo-mechanical stress screen

The steady FEniCSx temperature field is transferred to a steel-only axisymmetric elasticity model. The NeqSim handoff supplies internal pressure while water depth supplies external hydrostatic pressure. Thermal expansion is represented as an eigenstrain and therefore enters the right-hand-side load, while the bilinear form contains only mechanical stiffness. The result is a local thermo-mechanical screening calculation, not a design-code check.

In [ ]:
from scipy.interpolate import LinearNDInterpolator,NearestNDInterpolator

xy=Vt.tabulate_dof_coordinates()[:,:2]
linear_T=LinearNDInterpolator(xy,Th.x.array)
nearest_T=NearestNDInterpolator(xy,Th.x.array)
def transfer_temperature(xx):
    points=np.c_[xx[0],xx[1]]
    values=linear_T(points)
    missing=~np.isfinite(values)
    values[missing]=nearest_T(points[missing])
    return values

steel=mesh.create_rectangle(MPI.COMM_WORLD,np.array([[0.0,ri],[localL,rs]]),[80,8],cell_type=mesh.CellType.triangle)
steel_tags=tags_rect(steel,ri,rs)
ds=ufl.Measure('ds',domain=steel,subdomain_data=steel_tags)
dx=ufl.Measure('dx',domain=steel)
VT=fem.functionspace(steel,('Lagrange',1))
TT=fem.Function(VT)
TT.interpolate(transfer_temperature)
V=fem.functionspace(steel,('Lagrange',1,(2,)))
u,w=ufl.TrialFunction(V),ufl.TestFunction(V)
X=ufl.SpatialCoordinate(steel); r=X[1]
E,nu,alpha=207e9,.30,12e-6
mu=E/(2*(1+nu))
lam=E*nu/((1+nu)*(1-2*nu))
I3=ufl.Identity(3)

def eps(v):
    return ufl.as_tensor([[v[0].dx(0),.5*(v[0].dx(1)+v[1].dx(0)),0],[.5*(v[0].dx(1)+v[1].dx(0)),v[1].dx(1),0],[0,0,v[1]/r]])

def sigma_mechanical(v):
    ev=eps(v)
    return 2*mu*ev+lam*ufl.tr(ev)*I3

Tref=Tb+273.15
thermal_modulus=2*mu+3*lam
thermal_load=thermal_modulus*alpha*(TT-Tref)*ufl.tr(eps(w))*2*np.pi*r*dx
a=ufl.inner(sigma_mechanical(u),eps(w))*2*np.pi*r*dx
pi_pressure=Pb*1e5
po_pressure=rhosea*g*depth
pressure_load=ufl.dot(ufl.as_vector((0.0,pi_pressure)),w)*2*np.pi*r*ds(1)+ufl.dot(ufl.as_vector((0.0,-po_pressure)),w)*2*np.pi*r*ds(2)
Lform=pressure_load+thermal_load
fdim=steel.topology.dim-1
left=mesh.locate_entities_boundary(steel,fdim,lambda xx:np.isclose(xx[0],0.0))
dofs_z=fem.locate_dofs_topological(V.sub(0),fdim,left)
bc_z=fem.dirichletbc(PETSc.ScalarType(0.0),dofs_z,V.sub(0))
elastic_problem=LinearProblem(a,Lform,bcs=[bc_z],petsc_options_prefix='elastic_',petsc_options={'ksp_type':'preonly','pc_type':'lu'})
displacement=elastic_problem.solve()
assert elastic_problem.solver.getConvergedReason()>0
thermal_stress=thermal_modulus*alpha*(TT-Tref)*I3
S=sigma_mechanical(displacement)-thermal_stress
dev=S-ufl.tr(S)/3*I3
vm=ufl.sqrt(1.5*ufl.inner(dev,dev))
V0=fem.functionspace(steel,('Discontinuous Lagrange',0))
q=ufl.TrialFunction(V0); v0=ufl.TestFunction(V0)
vm_problem=LinearProblem(q*v0*dx,vm*v0*dx,petsc_options_prefix='vm_',petsc_options={'ksp_type':'preonly','pc_type':'lu'})
vmh=vm_problem.solve()
assert vm_problem.solver.getConvergedReason()>0
A_lame=(pi_pressure*ri**2-po_pressure*rs**2)/(rs**2-ri**2)
B_lame=ri**2*rs**2*(pi_pressure-po_pressure)/(rs**2-ri**2)
hoop_inner_pressure_only=(A_lame+B_lame/ri**2)/1e6
print(f'Lamé pressure-only inner hoop stress reference: {hoop_inner_pressure_only:.1f} MPa')
print(f'Maximum pressure + thermal von Mises screen: {float(vmh.x.array.max()/1e6):.1f} MPa')

## Model hierarchy and next use cases

Use NeqSim for the thermodynamic/process envelope and fast screening; use local FEM where geometry or material gradients matter. The companion `finite_element_methods_oil_gas_neqsim.ipynb` establishes the wider **Gmsh + scikit-fem + FEniCSx + PyVista** stack and includes porous diffusion and wellbore/formation heat transfer.

Natural extensions are buried-pipeline soil domains, separator/nozzle thermal stress, corrosion/electrochemistry PDEs and coupled geomechanics.